In [1]:
import scanpy
import anndata
import matplotlib
from matplotlib import pyplot
import hdf5plugin
import numpy
import scvelo
import seaborn
import pandas
import warnings
import cellrank
import gseapy

In [3]:
# Read input file
working_directory = "RNA Sequencing Data/"

# adata = scanpy.read_h5ad(working_directory + "/velocity_sdevelo.h5ad")

# Or read saved anndata objects
base_name = "cellrank_1.4"
adata = scanpy.read_h5ad(
    working_directory+f"/{base_name} cells.h5ad"
)
dead_end_adata = scanpy.read_h5ad(
    working_directory+f"/{base_name} dead end genes.h5ad"
)
mESC_adata = scanpy.read_h5ad(
    working_directory+f"/{base_name} mESC genes.h5ad"
)
driver_df = pandas.read_csv(
    working_directory+f"{base_name} driver genes.csv",
)

print(adata)

AnnData object with n_obs × n_vars = 12791 × 2000
    obs: 'barcode', 'batch', 'sample', 'group', 'day', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'initial_size_unspliced', 'initial_size_spliced', 'initial_size', 'n_counts', 'latent_time', 'sde_velocity_self_transition', 'manual_root', 'manual_end', 'sde_velocity_pseudotime', 'macrostates_fwd', 'term_states_fwd', 'term_states_fwd_probs', 'init_states_fwd', 'init_states_fwd_probs', 'clusters_gradients', 'fate_probabilities_Day 12 Control', 'fate_probabilities_mESC'
    var: 'ensemble_ids', 'gene_symbol', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'gene_count_corr', 'means', 'dispersions', 'dispersions_norm', 'highly_variable', 'fit_alpha', 'fit_beta', 'fit_gamma', 'fit_t_', 'fit_sigma_1', 'fit_sigma_2'
    uns: 'T_fwd_params', 'clusters_gradients_colors', 'coarse_fwd', 'eigendecomposition_fwd', 'init_states_fwd_colors', 'lineage_Day 12 Control_trend', 'lineage_mESC_t

In [4]:
# Show all possible libraries
names = gseapy.get_library_name(organism="Mouse")
print(names)

['ARCHS4_Cell-lines', 'ARCHS4_IDG_Coexp', 'ARCHS4_Kinases_Coexp', 'ARCHS4_TFs_Coexp', 'ARCHS4_Tissues', 'Achilles_fitness_decrease', 'Achilles_fitness_increase', 'Aging_Perturbations_from_GEO_down', 'Aging_Perturbations_from_GEO_up', 'Allen_Brain_Atlas_10x_scRNA_2021', 'Allen_Brain_Atlas_down', 'Allen_Brain_Atlas_up', 'Azimuth_2023', 'Azimuth_Cell_Types_2021', 'BioCarta_2013', 'BioCarta_2015', 'BioCarta_2016', 'BioPlanet_2019', 'BioPlex_2017', 'CCLE_Proteomics_2020', 'CM4AI_U2OS_Protein_Localization_Assemblies', 'COMPARTMENTS_Curated_2025', 'COMPARTMENTS_Experimental_2025', 'CORUM', 'COVID-19_Related_Gene_Sets', 'COVID-19_Related_Gene_Sets_2021', 'Cancer_Cell_Line_Encyclopedia', 'Carcinogenome', 'CellMarker_2024', 'CellMarker_Augmented_2021', 'ChEA_2013', 'ChEA_2015', 'ChEA_2016', 'ChEA_2022', 'Chromosome_Location', 'Chromosome_Location_hg19', 'ClinVar_2019', 'ClinVar_2025', 'DGIdb_Drug_Targets_2024', 'DSigDB', 'Data_Acquisition_Method_Most_Popular_Genes', 'DepMap_CRISPR_GeneDependency

## GO pathway

In [5]:
rank_data = driver_df[["gene_names", "Day 12 Control_corr"]].sort_values('Day 12 Control_corr', ascending=False).dropna()

# Run GSEA Prerank
pre_res = gseapy.prerank(
    rnk=rank_data, 
    gene_sets="GO_Biological_Process_2025",
    threads=16,
    min_size=5,
    max_size=1000,
    permutation_num=2000
)

2026-02-02 11:08:53,006 [WARNING] Duplicated values found in preranked stats: 0.15% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


In [6]:
# Enriched in dead end

pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "ES", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]] # Top Day 12

,Term,ES,NES,NOM p-val,FWER p-val,FDR q-val
0,Epidermis Development (GO:0008544),0.65148,1.832768,0.002571,0.4625,0.673351
1,Regulation of Trans-Synaptic Signaling (GO:009...,0.794156,1.804547,0.001188,0.593,0.504901
2,Regulation of Glycolytic Process (GO:0006110),0.816355,1.756093,0.00489,0.802,0.66826
3,Proteolysis Involved in Protein Catabolic Proc...,0.743789,1.755604,0.006105,0.803,0.504452
7,Antigen Processing and Presentation of Exogeno...,0.858342,1.711653,0.010753,0.935,0.505029
5,Antigen Processing and Presentation of Exogeno...,0.858342,1.711653,0.010753,0.935,0.505029
6,Antigen Processing and Presentation of Peptide...,0.858342,1.711653,0.010753,0.935,0.505029
10,Negative Regulation of Myeloid Leukocyte Diffe...,0.761694,1.639877,0.025862,0.9925,0.978021
11,Positive Regulation of Purine Nucleotide Catab...,0.871029,1.638283,0.009423,0.9925,0.795758
12,Positive Regulation of Glycolytic Process (GO:...,0.871029,1.638283,0.009423,0.9925,0.795758


In [7]:
# Enriched in iPSC path
pre_res.res2d.sort_values('NES', ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]  # Top mESC

,Term,NES,NOM p-val,FWER p-val,FDR q-val
4,Chromatin Organization (GO:0006325),-1.740148,0.003165,0.508,0.95249
8,Protein Polyubiquitination (GO:0000209),-1.69159,0.002494,0.7605,1.0
9,DNA Metabolic Process (GO:0006259),-1.652498,0.011735,0.9135,1.0
13,Male Meiotic Nuclear Division (GO:0007140),-1.636868,0.000833,0.9525,1.0
15,Transcription by RNA Polymerase II (GO:0006366),-1.626133,0.004149,0.9685,1.0
16,Transcription Initiation-Coupled Chromatin Rem...,-1.600658,0.006071,0.9885,1.0
17,Regulation of Viral Genome Replication (GO:004...,-1.594761,0.017964,0.991,1.0
18,Regulation of Insulin Secretion (GO:0050796),-1.592851,0.008584,0.992,1.0
24,Visual System Development (GO:0150063),-1.552392,0.006809,0.999,1.0
26,Negative Regulation of Bone Remodeling (GO:004...,-1.538783,0.006843,1.0,1.0


## GO molecular function

In [8]:
rank_data = driver_df[["gene_names", "Day 12 Control_corr"]].sort_values('Day 12 Control_corr', ascending=False).dropna()

# Run GSEA Prerank
pre_res = gseapy.prerank(
    rnk=rank_data, 
    gene_sets="GO_Molecular_Function_2025",
    threads=16,
    min_size=5,
    max_size=1000,
    permutation_num=2000
)

2026-02-02 11:09:29,168 [WARNING] Duplicated values found in preranked stats: 0.15% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


In [9]:
# Enriched in dead end

pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "ES", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]] # Top Day 12

,Term,ES,NES,NOM p-val,FWER p-val,FDR q-val
0,Chemokine Receptor Binding (GO:0042379),0.785248,1.921597,0.0,0.0225,0.028389
1,Cysteine-Type Endopeptidase Activity (GO:0004197),0.679228,1.680922,0.014252,0.457,0.374682
3,Exopeptidase Activity (GO:0008238),0.800233,1.647327,0.023284,0.577,0.36138
4,Chemokine Activity (GO:0008009),0.710391,1.628463,0.024643,0.6415,0.330893
6,Cysteine-Type Peptidase Activity (GO:0008234),0.613392,1.593236,0.02439,0.7675,0.376664
7,Carboxypeptidase Activity (GO:0004180),0.763355,1.589139,0.029817,0.781,0.326742
11,Monoatomic Cation Channel Activity (GO:0005261),0.663853,1.555057,0.040342,0.8685,0.377016
12,GTP Binding (GO:0005525),0.534418,1.522445,0.021978,0.933,0.429786
13,Phosphatase Binding (GO:0019902),0.674406,1.491764,0.068075,0.9625,0.476126
14,Guanyl Ribonucleotide Binding (GO:0032561),0.489552,1.46521,0.035088,0.9835,0.516413


In [10]:
# Enriched in iPSC path
pre_res.res2d.sort_values('NES', ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]  # Top mESC

,Term,NES,NOM p-val,FWER p-val,FDR q-val
2,mRNA Binding (GO:0003729),-1.662355,0.006809,0.282,0.370934
5,Iron Ion Binding (GO:0005506),-1.598965,0.004992,0.549,0.462834
8,2-Oxoglutarate-Dependent Dioxygenase Activity ...,-1.5827,0.007765,0.616,0.371886
9,Methylated Histone Binding (GO:0035064),-1.558144,0.009821,0.73,0.304556
10,Methylation-Dependent Protein Binding (GO:0140...,-1.558144,0.009821,0.73,0.304556
16,Protein Kinase C Binding (GO:0005080),-1.430195,0.064378,0.9865,0.924556
17,Guanyl-Nucleotide Exchange Factor Activity (GO...,-1.423662,0.079723,0.9915,0.841862
20,Single-Stranded DNA Binding (GO:0003697),-1.375777,0.094872,0.998,1.0
21,Ubiquitin Binding (GO:0043130),-1.359001,0.102322,0.999,1.0
23,GTPase Regulator Activity (GO:0030695),-1.328511,0.101404,1.0,1.0


## MSigDB_Hallmark_2020

In [11]:
rank_data = driver_df[["gene_names", "Day 12 Control_corr"]].sort_values('Day 12 Control_corr', ascending=False).dropna()

# Run GSEA Prerank
pre_res = gseapy.prerank(
    rnk=rank_data, 
    gene_sets='MSigDB_Hallmark_2020', # Or other libraries
    threads=16,
    min_size=5,
    max_size=1000,
)

2026-02-02 11:09:46,565 [WARNING] Duplicated values found in preranked stats: 0.15% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


In [12]:
# Enriched in dead end

pre_res.res2d.sort_values('NES', ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]] # Top Day 12

,Term,NES,NOM p-val,FWER p-val,FDR q-val
1,p53 Pathway,1.606321,0.002924,0.215,0.471552
2,Estrogen Response Early,1.566862,0.008086,0.284,0.317462
3,Cholesterol Homeostasis,1.501027,0.042184,0.433,0.370063
5,Androgen Response,1.43902,0.051213,0.555,0.410751
10,Allograft Rejection,1.363014,0.043732,0.738,0.535044
11,Estrogen Response Late,1.361683,0.043732,0.74,0.449274
12,Unfolded Protein Response,1.334365,0.142512,0.781,0.440256
13,IL-2/STAT5 Signaling,1.286976,0.059155,0.868,0.505433
15,mTORC1 Signaling,1.235128,0.136483,0.931,0.588099
16,Wnt-beta Catenin Signaling,1.231807,0.207254,0.935,0.536715


In [13]:
# Enriched in iPSC path
pre_res.res2d.sort_values('NES', ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]  # Top mESC

,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,G2-M Checkpoint,-1.633574,0.006154,0.136,0.073812
4,Spermatogenesis,-1.468485,0.060914,0.584,0.227791
6,DNA Repair,-1.435023,0.052901,0.678,0.197484
7,Oxidative Phosphorylation,-1.399611,0.067395,0.779,0.204328
8,Mitotic Spindle,-1.38439,0.061934,0.817,0.182331
9,E2F Targets,-1.378906,0.078947,0.831,0.159193
14,Interferon Alpha Response,-1.272851,0.124611,0.967,0.275207
26,Adipogenesis,-1.022899,0.424961,1.0,0.798308
30,Bile Acid Metabolism,-0.923083,0.587931,1.0,0.989322
31,Interferon Gamma Response,-0.839844,0.77259,1.0,1.0


## Cell types

In [14]:
rank_data = driver_df[["gene_names", "Day 12 Control_corr"]].sort_values('Day 12 Control_corr', ascending=False).dropna()

# Run GSEA Prerank
pre_res = gseapy.prerank(
    rnk=rank_data, 
    gene_sets='PanglaoDB_Augmented_2021', # Or other libraries
    threads=16,
    min_size=5,
    max_size=1000,
)

# Enriched in dead end

pre_res.res2d.sort_values('NES', ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]] # Top Day 12

2026-02-02 11:10:02,435 [WARNING] Duplicated values found in preranked stats: 0.15% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,Gastric Chief Cells,2.220668,0.0,0.0,0.0
1,Salivary Mucous Cells,2.177413,0.0,0.0,0.0
2,Cholangiocytes,2.036495,0.0,0.003,0.001335
3,Mammary Epithelial Cells,2.030373,0.0,0.004,0.001335
4,Keratinocytes,1.999058,0.0,0.011,0.003203
5,Luminal Epithelial Cells,1.982635,0.0,0.016,0.003782
6,Foveolar Cells,1.968645,0.0,0.018,0.003623
8,Microfold Cells,1.925072,0.0,0.023,0.004004
9,Sebocytes,1.910495,0.0,0.026,0.004152
10,Epithelial Cells,1.905622,0.0,0.029,0.004137


In [15]:
# Enriched in iPSC path
pre_res.res2d.sort_values('NES', ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]  # Top mESC

,Term,NES,NOM p-val,FWER p-val,FDR q-val
7,Pluripotent Stem Cells,-1.965991,0.0,0.002,0.00242
20,Epiblast Cells,-1.723253,0.001567,0.147,0.079066
23,Embryonic Stem Cells,-1.569442,0.002994,0.639,0.357948
24,Gamma Delta T Cells,-1.568614,0.015385,0.641,0.270075
27,Oxyphil Cells,-1.466755,0.054098,0.918,0.547491
30,Adrenergic Neurons,-1.414405,0.055456,0.969,0.700029
31,Ependymal Cells,-1.40723,0.076285,0.972,0.635755
32,Myocytes,-1.403557,0.047544,0.974,0.573631
36,Osteocytes,-1.338558,0.067976,0.996,0.828668
39,Microglia,-1.309889,0.071429,0.999,0.903691


Epithelial markers are strongly enriched among the control drivers

## ChipSeq targets, ChEA

In [16]:
rank_data = driver_df[["gene_names", "Day 12 Control_corr"]].sort_values('Day 12 Control_corr', ascending=False).dropna()

# Run GSEA Prerank
pre_res = gseapy.prerank(
    rnk=rank_data, 
    gene_sets='ChEA_2022', # Or other libraries
    threads=16,
    min_size=5,
    max_size=1000
)

2026-02-02 11:10:14,662 [WARNING] Duplicated values found in preranked stats: 0.15% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


In [17]:
# Enriched in dead end

pre_res.res2d.sort_values('NES', ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]] # Top Day 12

,Term,NES,NOM p-val,FWER p-val,FDR q-val
24,RARG 19884340 ChIP-ChIP MEFs Mouse,1.730881,0.0,0.221,0.318778
26,JARID2 20075857 ChIP-Seq MESCs Mouse,1.713658,0.0,0.248,0.183671
29,SUZ12 18974828 ChIP-Seq MESCs Mouse,1.686985,0.0,0.3,0.154408
38,MTF2 20144788 ChIP-Seq MESCs Mouse,1.639123,0.0,0.429,0.183671
42,SUZ12 27294783 Chip-Seq ESCs Mouse,1.612293,0.0,0.503,0.183546
49,SUZ12 20075857 ChIP-Seq MESCs Mouse,1.550556,0.0,0.699,0.267101
51,TP63 17297297 ChIP-ChIP HaCaT Human,1.54484,0.007712,0.72,0.24104
52,PITX1 30713093 ChIP-Seq Epithelial Human Tongu...,1.543706,0.027569,0.724,0.212
58,TP63 30713093 ChIP-Seq Epithelial Human Tongue...,1.507623,0.0,0.819,0.253473
62,JARID2 20064375 ChIP-Seq MESCs Mouse,1.493164,0.0,0.852,0.25689


In [18]:
# Enriched in iPSC path
pre_res.res2d.sort_values('NES', ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]  # Top mESC

,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,FOXM1 23109430 ChIP-Seq U2OS Human,-2.069772,0.0,0.0,0.0
1,SMAD1 18555785 ChIP-Seq MESCs Mouse,-2.06917,0.0,0.0,0.0
2,TCF3 18692474 ChIP-Seq MESCs Mouse,-2.031116,0.0,0.0,0.0
3,MYBL2 22936984 ChIP-ChIP MESCs Mouse,-2.028174,0.0,0.0,0.0
4,NACC1 18358816 ChIP-ChIP MESCs Mouse,-2.026743,0.0,0.0,0.0
5,NANOG 18700969 ChIP-ChIP MESCs Mouse,-1.998373,0.0,0.0,0.0
6,NANOG 18555785 ChIP-Seq MESCs Mouse,-1.925797,0.0,0.004,0.000518
7,SOX2 18692474 ChIP-Seq MESCs Mouse,-1.921982,0.0,0.004,0.000454
8,POU5F1 18358816 ChIP-ChIP MESCs Mouse,-1.919393,0.0,0.005,0.000504
9,POU5F1 18700969 ChIP-ChIP MESCs Mouse,-1.909834,0.0,0.006,0.000544


## Enriched targets from ChipSeq, Ensembl

In [19]:
rank_data = driver_df[["gene_names", "Day 12 Control_corr"]].sort_values('Day 12 Control_corr', ascending=False).dropna()

# Run GSEA Prerank
pre_res = gseapy.prerank(
    rnk=rank_data, 
    gene_sets='ENCODE_TF_ChIP-seq_2015', # Or other libraries
    threads=16,
    min_size=5,
    max_size=1000
)

2026-02-02 11:11:13,634 [WARNING] Duplicated values found in preranked stats: 0.15% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


In [20]:
# Enriched in dead end

pre_res.res2d.sort_values('NES', ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]] # Top Day 12

,Term,NES,NOM p-val,FWER p-val,FDR q-val
1,FOSL1 C2C12 mm9,1.96268,0.0,0.019,0.021294
8,SMARCC1 HeLa-S3 hg19,1.723216,0.0,0.208,0.135609
21,ZEB1 HepG2 hg19,1.635744,0.01039,0.412,0.216301
23,TAL1 G1E-ER4 mm9,1.616329,0.0,0.468,0.200051
25,ATF3 K562 hg19,1.570052,0.0,0.611,0.257993
31,EP300 ECC-1 hg19,1.540566,0.003077,0.698,0.279623
42,SMARCC2 HeLa-S3 hg19,1.498257,0.016393,0.799,0.350949
46,MAX myocyte mm9,1.47301,0.009375,0.86,0.381329
48,RAD21 ECC-1 hg19,1.462772,0.010239,0.878,0.370713
52,JUND GM12878 hg19,1.447306,0.033766,0.904,0.381609


In [21]:
# Enriched in iPSC path
pre_res.res2d.sort_values('NES', ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]  # Top mESC

,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,E2F4 MEL cell line mm9,-2.037147,0.0,0.0,0.0
2,FOXM1 ECC-1 hg19,-1.840812,0.0,0.028,0.013301
3,IRF3 GM12878 hg19,-1.807172,0.0,0.046,0.015201
4,E2F4 CH12.LX mm9,-1.804058,0.0,0.048,0.011876
5,NANOG H1-hESC hg19,-1.740666,0.001751,0.147,0.033442
6,E2F4 HeLa-S3 hg19,-1.736897,0.0,0.155,0.029926
7,SP2 K562 hg19,-1.735902,0.0,0.158,0.026194
9,NFYA GM12878 hg19,-1.717984,0.0,0.213,0.031945
10,FOS K562 hg19,-1.694223,0.0,0.281,0.040535
11,FOXM1 MCF-7 hg19,-1.691418,0.003205,0.289,0.037812


## Test if specific gene sets are enriched

In [33]:
# Get libraries

panglao_lib = gseapy.get_library(name='PanglaoDB_Augmented_2021', organism='Mouse')
go_lib = gseapy.get_library(name='GO_Biological_Process_2025', organism='Mouse')
go_molecular_function = gseapy.get_library(name='GO_Molecular_Function_2025', organism='Mouse')
hallmark_lib = gseapy.get_library(name='MSigDB_Hallmark_2020', organism='Mouse')

In [34]:
target_sets = {
    "Apoptosis Activation": go_lib["Positive Regulation of Apoptotic Process (GO:0043065)"],
    "Apoptosis Inhibition": go_lib["Negative Regulation of Apoptotic Process (GO:0043066)"],
    "Cell Cycle Inhibition": go_lib["Negative Regulation of Cell Cycle (GO:0045786)"],
    "Cell Cycle Activation": go_lib["Positive Regulation of Cell Cycle (GO:0045787)"],
    "Apoptosis Signaling Inhibition": go_lib["Negative Regulation of Apoptotic Signaling Pathway (GO:2001234)"],
    "Apoptosis Signaling Activation": go_lib["Positive Regulation of Apoptotic Signaling Pathway (GO:2001235)"],
    "Stem Cell Proliferation": go_lib["Positive Regulation of Stem Cell Proliferation (GO:2000648)"],
    "Stem Cell Inhibited Proliferation": go_lib["Negative Regulation of Stem Cell Population Maintenance (GO:1902455)"],
    "Stem Cell Inhibited Differentiation": go_lib["Negative Regulation of Stem Cell Differentiation (GO:2000737)"],
    "Stem Cell Differentiation": go_lib["Positive Regulation of Stem Cell Differentiation (GO:2000738)"],
    "DNA Damage Response": go_lib["DNA Damage Response (GO:0006974)"],
    "DNA_Repair": hallmark_lib['DNA Repair'],
    "P53_Pathway": hallmark_lib['p53 Pathway'],
    "Keratinocytes": panglao_lib["Keratinocytes"],
    "Epithelial cells": panglao_lib["Epithelial Cells"]
}

rank_data = driver_df[["gene_names", "Day 12 Control_corr"]].sort_values('Day 12 Control_corr', ascending=False).dropna()

# Run GSEA Prerank
pre_res = gseapy.prerank(
    rnk=rank_data, 
    gene_sets=target_sets,
    threads=16,
    min_size=5,
    max_size=2000,
    permutation_num=1000,
    seed=0
)

pre_res.res2d.sort_values("NES", ascending=False).head(20)

2026-02-02 11:19:48,524 [WARNING] Duplicated values found in preranked stats: 0.15% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
0,prerank,Keratinocytes,0.624645,2.0251,0.0,0.001222,0.001,22/41,17.03%,Lgals7;Lad1;Pkp1;Perp;Ovol1;Anxa8;S100a14;Krt1...
1,prerank,Epithelial cells,0.541295,1.924159,0.0,0.001222,0.002,32/67,19.08%,Lad1;Crb3;Mal2;Ovol1;Krt8;Dgat2;Gpx2;Misp;S100...
2,prerank,P53_Pathway,0.465473,1.60209,0.003289,0.032179,0.078,24/60,18.48%,Ccnd2;Ifi30;Nupr1;Perp;Cdkn1a;Sat1;Gpx2;Krt17;...
3,prerank,Apoptosis Signaling Activation,0.654014,1.540293,0.045024,0.038187,0.118,5/11,16.88%,Nupr1;Bbc3;App;Tnfrsf12a;Ddit3
7,prerank,Apoptosis Inhibition,0.210775,0.79448,0.944444,0.966599,0.978,18/107,15.12%,Ccnd2;Nupr1;Prkaa2;Cd44;Myc;Anxa5;Tmbim1;Ddah2...
10,prerank,Cell Cycle Inhibition,0.281041,0.694447,0.905,0.916497,0.988,1/14,1.40%,Nupr1
12,prerank,Apoptosis Signaling Inhibition,-0.238535,-0.560375,0.964346,0.984178,1.0,4/14,33.20%,Tcf7l2;Bmp4;Thbs1;Clu
11,prerank,Stem Cell Proliferation,-0.283705,-0.572438,0.962838,1.0,1.0,1/8,7.46%,Nanog
9,prerank,Cell Cycle Activation,-0.29015,-0.707803,0.871545,1.0,1.0,2/16,4.96%,Tcf7l1;Cited2
8,prerank,Apoptosis Activation,-0.241989,-0.775089,0.896603,1.0,1.0,29/63,37.66%,Dlc1;Rest;Gadd45b;Top2a;Bnip3;Tsc22d1;Sfrp1;Ec...
